# Data preprocessing

In [1]:
import numpy as np
import uuid
from tqdm.auto import tqdm

In [2]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 70.6 MB/s eta 0:00:00:00:0100:01


In [3]:
import re
import fitz  # PyMuPDF

def clean_text(t):
    t = re.sub(r'-\n', '', t) 
    t = re.sub(r'\s+\n', '\n', t)
    t = re.sub(r'\n{3,}', '\n\n', t)
    t = re.sub(r'Page \d+', '', t)
    return t.strip()

doc = fitz.open("/kaggle/input/casml-dataset2/Dataset_RAG (1)/book.pdf")
pages = []
page_start_positions = []  # сохраняем, где начинается каждая страница в общем тексте
text = ""
for i, page in enumerate(doc):
    page_start_positions.append(len(text))
    page_text = page.get_text("text")
    page_text = clean_text(page_text)
    text += page_text + "\n"
    pages.append(page_text)

with open("textbook.txt", "w", encoding="utf-8") as f:
    f.write(text)

print(f"Извлечено {len(pages)} страниц. Общая длина текста: {len(text):,} символов.")


Извлечено 753 страниц. Общая длина текста: 2,254,755 символов.


In [4]:
# print(text[:100000])

# Model

In [4]:
from transformers import pipeline
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)


model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype="auto",
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

2025-11-10 09:51:23.955617: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762768284.205633      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762768284.278759      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
)

print(generation_pipeline(messages, max_new_tokens=256, do_sample=True, temperature=0.3, top_p=0.9)[0]['generated_text'][-1]['content'])

Device set to use cuda:0


A large language model is an artificial intelligence system that can generate human-like text based on the input it receives from users or other sources of data. These models use deep learning techniques, such as neural networks and recurrent neural networks (RNNs), to analyze vast amounts of text data and learn patterns and relationships between words and phrases.

Large language models have been trained on massive datasets containing billions of words of text, including books, articles, web pages, and social media posts. They can be used for various tasks, such as generating creative writing, translating languages, answering questions, and even playing games like chess or Go.

One of the most famous examples of a large language model is GPT-3, developed by OpenAI in 2020. It was trained on over 1 trillion tokens of text data and has achieved impressive results in many natural language processing tasks. Other notable large language models include BERT, T5, and XLNet, which have also b

# vector bd

In [6]:
# import torch.nn.functional as F
# from transformers import AutoModel

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("intfloat/multilingual-e5-large", model_kwargs={'torch_dtype': torch.float16})

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [7]:
!pip install langchain-qdrant

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.3/337.3 kB 17.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.72
    Uninstalling langchain-core-0.3.72:
      Successfully uninstalled langchain-core-0.3.72
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.4 which is incompatible.
langchain-text-splitters 0.3.9 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.4 which is incompatible.


In [8]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="psycology_e5",
    on_disk_payload=True,
    vectors_config=models.VectorParams(
        size=1024,
        distance=models.Distance.COSINE,
        on_disk=True
    ),
)

True

# chunk-split

In [9]:
!pip install langchain-community

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.9
    Uninstalling langchain-text-splitters-0.3.9:
      Successfully uninstalled langchain-text-splitters-0.3.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.4 which is incompatible.
langchain 0.3.27 requires langchain-text-splitters<1.0.0,>=0.3.9, but you have langchain-text-splitters 1.0.0 which is incompatible.


In [10]:
def find_page_for_chunk(start_index, page_starts):
    """Находит, на какой странице начинается чанк по его позиции в тексте."""
    if start_index is None or start_index < 0:
        return 1  # fallback — если не нашли позицию, считаем первой страницей
    for i, p_start in enumerate(page_starts):
        if start_index < p_start:
            return max(1, i)  # страницы нумеруются с 1
    return len(page_starts) 

In [56]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150, separators=["\n\n", "\n", " ", ""])

In [12]:
chunks = []
last_pos = 0
for chunk in text_splitter.split_text(text):
    if not chunk.strip():  # пропускаем пустые чанки
        continue

    start_index = text.find(chunk, last_pos)
    if start_index == -1:
        # если find не нашёл (например, повтор текста), пробуем искать с начала
        start_index = text.find(chunk)
        if start_index == -1:
            # если вообще не нашли — пропускаем этот чанк
            print(f"Не найден чанк{chunk} в тексте, пропущен.")
            continue

    page = find_page_for_chunk(start_index, page_start_positions)
    chunks.append({"text": chunk, "page": page})
    last_pos = start_index + len(chunk)

print(f"Получено {len(chunks)} чанков.")

Получено 1567 чанков.


In [13]:
vectors = embedding_model.encode([chunk["text"] for chunk in chunks],
                                 batch_size=32, device=device, normalize_embeddings=True, show_progress_bar=True).tolist()

Batches:   0%|          | 0/49 [00:00<?, ?it/s]

In [14]:
batch_size = 64

for i in tqdm(range(0, len(vectors), batch_size)):
    batch_points = [
        models.PointStruct(
            id=str(uuid.uuid4()),
            vector=vectors[j],
            payload={
                'text': chunks[j]["text"],
                'page': chunks[j]["page"],
            }
        )
        for j in range(i, min(i + batch_size, len(vectors)))
    ]
    client.upsert(collection_name='psycology_e5', points=batch_points)


  0%|          | 0/25 [00:00<?, ?it/s]

# LLM-answer

In [41]:
def llm_answer(query, context, temperature=0.4):
    prompt = f"""Текст из учебника психологии:
{context}

Вопрос:
{query}"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a very skeptical scientist in the field of psychology. You will receive a context consisting of text clippings from a book on the desired topic. Your task is to answer the user as accurately and honestly as possible. Make sure that the answer is detailed, specific, and directly related to the question. Do not add information that is not directly supported by the provided clippings from the book. If there is no direct answer in the text, tell me about it honestly."
            ),
        },
        {"role": "user", "content": prompt},
    ]

    output = generation_pipeline(
        messages,
        max_new_tokens=512,
        do_sample=True,
        temperature=temperature,
        top_p=0.8,
    )

    # Проверяем структуру вывода, т.к. она может отличаться между версиями Transformers
    if isinstance(output[0]["generated_text"], list):
        # новый формат: список сообщений
        return output[0]["generated_text"][-1]["content"]
    elif isinstance(output[0]["generated_text"], str):
        # старый формат: просто строка
        return output[0]["generated_text"]
    else:
        # fallback
        return str(output[0])


# Submission

In [51]:
def semantic_search(client, query, limit=3, collection_name="psycology_e5"):
    """
    Выполняет семантический поиск в коллекции Qdrant.

    Аргументы:
        client: экземпляр QdrantClient
        query: текстовый запрос пользователя
        limit: количество возвращаемых чанков (по умолчанию 10)
        collection_name: имя коллекции (по умолчанию 'psycology_e5')

    Возвращает:
        Список словарей формата:
        [
            {
                "text": "...",          # сам чанк
                "page": 123,            # страница, если есть в payload
                "score": 0.87           # косинусная близость
            },
            ...
        ]
    """
    # Кодируем запрос
    query_vector = embedding_model.encode(
        query,
        normalize_embeddings=True,
        device=device
    ).tolist()

    # Делаем запрос в Qdrant
    hits = client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        limit=limit,
        score_threshold=0.2
    )

    # Безопасно обрабатываем результаты
    if not hits:
        print("Предупреждение: по запросу ничего не найдено.")
        return []

    # Собираем полезную информацию из результатов
    results = []
    for hit in hits:
        payload = hit.payload or {}
        results.append({
            "text": payload.get("text", ""),
            "page": payload.get("page", None),
            "score": hit.score
        })

    return results


In [38]:
import json
import pandas as pd

queries = json.load(open("/kaggle/input/casml-dataset2/Dataset_RAG (1)/queries.json"))

results = []
for q in tqdm(queries):
    query = q["question"]
    query_id = q["query_id"]

    relevant_chunks = semantic_search(client, query, limit=5)
    context = "\n\n".join([chunk["text"] for chunk in relevant_chunks])
    pages = sorted({chunk["page"] for chunk in relevant_chunks if chunk.get("page") is not None})
    # pages = sorted(list({chunk["page"] for chunk in relevant_chunks}))
    references = json.dumps({"pages": pages})

    answer = llm_answer(query, context)

    results.append({
        "ID": query_id,
        "context": context,
        "answer": answer,
        "references": references
    })

df = pd.DataFrame(results)
df.to_csv("submission_basic_rag9.csv", index=False)
print("submission_basic_rag9.csv saved")


  0%|          | 0/50 [00:00<?, ?it/s]

TypeError: semantic_search() got an unexpected keyword argument 'limit'

# Reranker

In [33]:
from sentence_transformers import CrossEncoder
import numpy as np
import torch

# Загружаем кросс-энкодер
scorer = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)

def semantic_search(
    client,
    query,
    collection_name="psycology_e5",
    top_k=15,
    confidence_threshold=2.0,  # напрямую на "сырые" оценки
    show_confidences=True
):
    # 1️⃣ Кодируем запрос
    query_vector = embedding_model.encode(query, normalize_embeddings=True, device=device).tolist()

    # 2️⃣ Получаем результаты из Qdrant
    res = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k,
        score_threshold=0.2
    )

    # 3️⃣ Унификация формата
    if hasattr(res, "points"):
        hits = res.points
    elif isinstance(res, (list, tuple)) and len(res) > 0:
        if hasattr(res[0], "payload"):
            hits = res
        elif hasattr(res[0], "points"):
            hits = res[0].points
        else:
            hits = [x for x in res if hasattr(x, "payload")]
    else:
        hits = []

    if not hits:
        print("⚠️ По запросу ничего не найдено.")
        return []

    # 4️⃣ Собираем кандидатов
    candidates = []
    for hit in hits:
        try:
            payload = getattr(hit, "payload", None) or hit.get("payload", {})
        except Exception:
            payload = {}
        candidates.append({
            "text": payload.get("text", ""),
            "page": payload.get("page", None),
            "score": float(getattr(hit, "score", 0) or hit.get("score", 0))
        })

    # 5️⃣ Оценка через CrossEncoder (без нормализации)
    pairs = [(query, c["text"]) for c in candidates]
    with torch.no_grad():
        confidences = scorer.predict(pairs)

    confidences = np.array(confidences).astype(float).reshape(-1)
    for c, conf in zip(candidates, confidences.tolist()):
        c["confidence"] = conf

    # 6️⃣ Сортировка по убыванию
    sorted_candidates = sorted(candidates, key=lambda x: x["confidence"], reverse=True)

    # 7️⃣ Фильтрация по порогу
    filtered = [c for c in sorted_candidates if c["confidence"] >= confidence_threshold]

    # 8️⃣ Если никто не прошёл — берём один лучший
    if not filtered:
        best = sorted_candidates[0]
        filtered = [best]
        print(f"⚠️ Нет кандидатов выше порога {confidence_threshold:.2f}. "
              f"Взято fallback: уверенность {best['confidence']:.2f}")
    else:
        print(f"✅ Отобрано {len(filtered)} из {len(sorted_candidates)} кандидатов "
              f"(порог {confidence_threshold:.2f})")

    # 9️⃣ Отладочная печать
    if show_confidences:
        print(f"\nЗапрос: {query}")
        for c in sorted_candidates:
            clean_text = c["text"].replace("\n", " ")[:80]
            print(f"Уверенность {c['confidence']:.2f} | {clean_text}...")

    return filtered


In [34]:
import json
import pandas as pd

queries = json.load(open("/kaggle/input/casml-dataset2/Dataset_RAG (1)/queries.json"))

results = []
for q in tqdm(queries):
    query = q["question"]
    query_id = q["query_id"]

    relevant_chunks = semantic_search(client, query)
    context = "\n\n".join([chunk["text"] for chunk in relevant_chunks])
    pages = sorted({chunk["page"] for chunk in relevant_chunks if chunk.get("page") is not None})
    # pages = sorted(list({chunk["page"] for chunk in relevant_chunks}))
    references = json.dumps({"pages": pages})

    answer = llm_answer(query, context)

    results.append({
        "ID": query_id,
        "context": context,
        "answer": answer,
        "references": references
    })

df = pd.DataFrame(results)
df.to_csv("submission_basic_rag10.csv", index=False)
print("submission_basic_rag10.csv saved")


  0%|          | 0/50 [00:00<?, ?it/s]

✅ Отобрано 4 из 15 кандидатов (порог 2.00)

Запрос: What is the scientific method in psychology?
Уверенность 4.98 | education institutions, increasing the number of Black Americans who went on to ...
Уверенность 4.64 | 1 Introduction to Psychology 2002). Nash was the subject of the 2001 movie A Bea...
Уверенность 4.47 | published or presented at research conferences so that others can replicate or b...
Уверенность 3.32 | Summary 1.1 What Is Psychology? Psychology is defined as the scientific study of...
Уверенность 1.87 | hypotheses to test specific aspects of a theory. A hypothesis is a testable pred...
Уверенность -0.10 | understanding behavior, as well as the cognitive (mental) and physiological (bod...
Уверенность -0.14 | lifelong process that can be studied scientifically across three developmental d...
Уверенность -1.49 | beliefs. This particular method can provide large amounts of information in rela...
Уверенность -2.62 | humanism perspective within psychology that emphasizes t

# подаем на выход не отфильтрованный, а с перестановками (вперед - в конец и тд)

In [54]:
from sentence_transformers import CrossEncoder
import numpy as np
import torch

# Загружаем кросс-энкодер
scorer = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)

def semantic_search(
    client,
    query,
    collection_name="psycology_e5",
    top_k=15,
    confidence_threshold=1.0,
    show_confidences=True
):
    # 1️⃣ Кодируем запрос
    query_vector = embedding_model.encode(query, normalize_embeddings=True, device=device).tolist()

    # 2️⃣ Получаем результаты из Qdrant
    res = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k,
        score_threshold=0.2
    )

    # 3️⃣ Унификация формата
    if hasattr(res, "points"):
        hits = res.points
    elif isinstance(res, (list, tuple)) and len(res) > 0:
        if hasattr(res[0], "payload"):
            hits = res
        elif hasattr(res[0], "points"):
            hits = res[0].points
        else:
            hits = [x for x in res if hasattr(x, "payload")]
    else:
        hits = []

    if not hits:
        print("⚠️ По запросу ничего не найдено.")
        return []

    # 4️⃣ Собираем кандидатов
    candidates = []
    for hit in hits:
        try:
            payload = getattr(hit, "payload", None) or hit.get("payload", {})
        except Exception:
            payload = {}
        candidates.append({
            "text": payload.get("text", ""),
            "page": payload.get("page", None),
            "score": float(getattr(hit, "score", 0) or hit.get("score", 0))
        })

    # 5️⃣ CrossEncoder оценка
    pairs = [(query, c["text"]) for c in candidates]
    with torch.no_grad():
        confidences = scorer.predict(pairs)

    confidences = np.array(confidences).astype(float).reshape(-1)
    for c, conf in zip(candidates, confidences.tolist()):
        c["confidence"] = conf

    # 6️⃣ Сортировка по убыванию уверенности
    sorted_candidates = sorted(candidates, key=lambda x: x["confidence"], reverse=True)

    # 7️⃣ Фильтрация по порогу
    filtered = [c for c in sorted_candidates if c["confidence"] >= confidence_threshold]

    if not filtered:
        best = sorted_candidates[0]
        filtered = [best]
        print(f"⚠️ Нет кандидатов выше порога {confidence_threshold:.2f}. "
              f"Взято fallback: уверенность {best['confidence']:.2f}")
    else:
        print(f"✅ Отобрано {len(filtered)} из {len(sorted_candidates)} кандидатов "
              f"(порог {confidence_threshold:.2f})")

    # 8️⃣ Перестановка: первый, третий, пятый, затем четвёртый, второй (зеркально)
    n = len(filtered)
    first_half = filtered[::2]          # индексы 0, 2, 4, ...
    second_half = filtered[1::2][::-1]  # индексы 1, 3, 5, ... но в обратном порядке
    reordered = first_half + second_half

    # 9️⃣ Для отладки — печать
    if show_confidences:
        print(f"\nЗапрос: {query}")
        for i, c in enumerate(reordered):
            clean_text = c["text"].replace("\n", " ")[:80]
            print(f"{i+1:>2}. Уверенность {c['confidence']:.2f} | {clean_text}...")
    print("\n")

    return reordered


In [55]:
import json
import pandas as pd

queries = json.load(open("/kaggle/input/casml-dataset2/Dataset_RAG (1)/queries.json"))

results = []
for q in tqdm(queries):
    query = q["question"]
    query_id = q["query_id"]

    relevant_chunks = semantic_search(client, query, confidence_threshold=0.0)
    context = "\n\n".join([chunk["text"] for chunk in relevant_chunks])
    pages = sorted({chunk["page"] for chunk in relevant_chunks if chunk.get("page") is not None})
    # pages = sorted(list({chunk["page"] for chunk in relevant_chunks}))
    references = json.dumps({"pages": pages})

    answer = llm_answer(query, context, temperature=0.05)

    results.append({
        "ID": query_id,
        "context": context,
        "answer": answer,
        "references": references
    })

df = pd.DataFrame(results)
df.to_csv("submission_basic_rag11.csv", index=False)
print("submission_basic_rag11.csv saved")


  0%|          | 0/50 [00:00<?, ?it/s]

✅ Отобрано 5 из 15 кандидатов (порог 0.00)

Запрос: What is the scientific method in psychology?
 1. Уверенность 4.98 | education institutions, increasing the number of Black Americans who went on to ...
 2. Уверенность 4.47 | published or presented at research conferences so that others can replicate or b...
 3. Уверенность 1.87 | hypotheses to test specific aspects of a theory. A hypothesis is a testable pred...
 4. Уверенность 3.32 | Summary 1.1 What Is Psychology? Psychology is defined as the scientific study of...
 5. Уверенность 4.64 | 1 Introduction to Psychology 2002). Nash was the subject of the 2001 movie A Bea...


⚠️ Нет кандидатов выше порога 0.00. Взято fallback: уверенность -1.87

Запрос: What are the basic parts of a neuron?
 1. Уверенность -1.87 | FIGURE 3.8 This illustration shows a prototypical neuron, which is being myelina...


✅ Отобрано 7 из 15 кандидатов (порог 0.00)

Запрос: What are the stages of sleep?
 1. Уверенность 7.32 | modification of work by Ryan Vaars

# Перефразирование вопроса

In [58]:
!pip install rank_bm25

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [61]:
# ==========================================
# 🔄 Improved Dense Retrieval + Query Expansion
# ==========================================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# --- 1️⃣ Модель для query expansion (перефразирование) ---
qe_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
qe_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(device)

def expand_query(query, n=3):
    """Перефразирует запрос несколькими способами для лучшего покрытия контекста"""
    prompt = f"Generate {n} alternative phrasings for the question: '{query}'"
    inputs = qe_tokenizer(prompt, return_tensors="pt").to(device)
    outputs = qe_model.generate(**inputs, max_new_tokens=64, num_return_sequences=n, temperature=0.8, do_sample=True)
    return [qe_tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

# --- 2️⃣ Новый reranker ---
scorer = CrossEncoder("BAAI/bge-reranker-base", device=device)

def advanced_semantic_search(client, query, collection_name="psycology_e5", top_k=20):
    """Semantic retrieval с query expansion и reranking"""
    expanded = expand_query(query, n=3)
    all_hits = []

    # для каждого перефраза — ищем топ-K
    for q_exp in expanded + [query]:
        q_vec = embedding_model.encode(q_exp, normalize_embeddings=True, device=device).tolist()
        res = client.query_points(collection_name=collection_name, query=q_vec, limit=top_k)
        all_hits += [
            {"text": p.payload["text"], "page": p.payload["page"], "score": float(p.score)}
            for p in res.points
        ]

    # удаляем дубликаты
    unique = {}
    for h in all_hits:
        key = h["text"][:200]
        if key not in unique:
            unique[key] = h

    hits = list(unique.values())

    # reranking
    pairs = [(query, h["text"]) for h in hits]
    with torch.no_grad():
        rerank_scores = scorer.predict(pairs)

    for h, s in zip(hits, rerank_scores):
        h["confidence"] = float(s)

    hits = sorted(hits, key=lambda x: x["confidence"], reverse=True)[:10]
    return hits


# ==========================================
# 🧠 Улучшенная генерация ответа
# ==========================================
def llm_answer_v3(query, context, temperature=0.3):
    prompt = f"""
You are an expert in psychology explaining concepts from a textbook.
Use only the provided context to answer precisely. 
Include detailed reasoning and refer to psychological terminology as written in the book.
If unsure, say the text does not contain the answer.

Context:
{context}

Question:
{query}

Answer:
"""

    messages = [
        {"role": "system", "content": "You are a precise and insightful psychology tutor."},
        {"role": "user", "content": prompt},
    ]

    output = generation_pipeline(
        messages,
        max_new_tokens=512,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
    )

    if isinstance(output[0]["generated_text"], list):
        return output[0]["generated_text"][-1]["content"]
    elif isinstance(output[0]["generated_text"], str):
        return output[0]["generated_text"]
    else:
        return str(output[0])


# ==========================================
# 🧪 Генерация сабмита
# ==========================================
results_v3 = []

for q in tqdm(queries):
    query = q["question"]
    query_id = q["query_id"]

    relevant_chunks = advanced_semantic_search(client, query)
    context = "\n\n".join([c["text"] for c in relevant_chunks])
    pages = sorted({c["page"] for c in relevant_chunks if c.get("page")})
    references = json.dumps({"pages": pages})

    answer = llm_answer_v3(query, context, temperature=0.05)

    results_v3.append({
        "ID": query_id,
        "context": context,
        "answer": answer,
        "references": references
    })

df_v3 = pd.DataFrame(results_v3)
df_v3.to_csv("submission_dense_qe_reranker.csv", index=False)
print("✅ submission_dense_qe_reranker.csv saved successfully!")


100%|██████████| 50/50 [09:58<00:00, 11.96s/it]

✅ submission_dense_qe_reranker.csv saved successfully!


In [63]:
# ==========================================
# 🚀 Improved Dense RAG System (v4)
# ==========================================
import json
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import CrossEncoder

# ==========================================
# 1️⃣ Query Expansion (контролируемое, тематическое)
# ==========================================
qe_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
qe_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(device)

def expand_query(query, n=3):
    """Перефразирует запрос с сохранением психологической терминологии"""
    prompt = f"""
You are an expert in psychology. 
Rephrase the following question {n} different ways, keeping key psychological terminology unchanged.
Focus on preserving meaning, not just rewording.

Question: {query}
"""
    inputs = qe_tokenizer(prompt, return_tensors="pt").to(device)
    outputs = qe_model.generate(
        **inputs,
        max_new_tokens=96,
        num_return_sequences=n,
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
    )
    paraphrases = [qe_tokenizer.decode(o, skip_special_tokens=True).strip() for o in outputs]
    filtered = list({p for p in paraphrases if len(p.split()) > 3})
    return filtered[:n]


# ==========================================
# 2️⃣ Cross-Encoder Reranker (улучшенный)
# ==========================================
scorer = CrossEncoder("BAAI/bge-reranker-base", device=device)

# ==========================================
# 3️⃣ Adaptive Top-K Retrieval + Query Fusion
# ==========================================
MAX_CONTEXT_LEN = 5500  # ограничение длины контекста (~3500 токенов для Qwen)

def dynamic_top_k(query):
    """Динамически выбирает число возвращаемых фрагментов в зависимости от длины запроса"""
    length = len(query.split())
    if length < 6:
        return 15
    elif length < 15:
        return 25
    else:
        return 35

def advanced_semantic_search(client, query, collection_name="psycology_e5"):
    """Semantic retrieval с query expansion, fusion и reranking"""
    expanded = expand_query(query, n=3)
    all_hits = []
    top_k = dynamic_top_k(query)

    # ищем по каждому перефразу
    for q_exp in expanded + [query]:
        q_vec = embedding_model.encode(q_exp, normalize_embeddings=True, device=device).tolist()
        res = client.query_points(collection_name=collection_name, query=q_vec, limit=top_k)
        for p in res.points:
            all_hits.append({"text": p.payload["text"], "page": p.payload["page"], "score": float(p.score)})

    # убираем дубликаты
    seen = set()
    unique_hits = []
    for h in all_hits:
        key = h["text"][:160]
        if key not in seen:
            seen.add(key)
            unique_hits.append(h)

    # reranking
    pairs = [(query, h["text"]) for h in unique_hits]
    with torch.no_grad():
        scores = scorer.predict(pairs)
    for h, s in zip(unique_hits, scores):
        h["confidence"] = float(s)

    hits = sorted(unique_hits, key=lambda x: x["confidence"], reverse=True)

    # Query Fusion: набираем контекст до лимита
    context, pages, total_len = [], set(), 0
    for h in hits:
        t = h["text"].strip()
        if total_len + len(t) > MAX_CONTEXT_LEN:
            break
        context.append(t)
        pages.add(h["page"])
        total_len += len(t)

    return {"context": "\n\n".join(context), "pages": sorted(pages)}


# ==========================================
# 4️⃣ Улучшенная генерация ответа (Qwen)
# ==========================================
from transformers import GenerationConfig

def llm_answer_v4(query, context, temperature=0.3, top_p=0.9, top_k=40):
    """Генерация ответа с полным контролем sampling-параметров"""

    # Промпт
    prompt = f"""
You are a psychology tutor helping students understand the textbook.
Use only the provided context to answer precisely. 
If the context does not contain the answer, say so directly.

Answer requirements:
1. Be detailed but concise.
2. Explain reasoning based on the provided context.
3. Mention key psychological terms exactly as in the text.

Context:
{context}

Question:
{query}

Answer:
"""

    # Формируем system-style подсказку
    full_prompt = f"<|im_start|>system\nYou are a precise and rigorous psychology expert who always grounds answers in source material.<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

    generation_config = GenerationConfig(
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        do_sample=True,
        max_new_tokens=512,
    )

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            generation_config=generation_config,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Декодируем текст
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Убираем prompt из вывода (чтобы оставить только ответ)
    answer = text.split("Answer:")[-1].strip()
    return answer


# ==========================================
# 5️⃣ Генерация сабмита
# ==========================================
results_v4 = []

for q in tqdm(queries):
    query = q["question"]
    query_id = q["query_id"]

    search_res = advanced_semantic_search(client, query)
    context, pages = search_res["context"], search_res["pages"]

    answer = llm_answer_v4(query, context)
    references = json.dumps({"pages": pages})

    results_v4.append({
        "ID": query_id,
        "context": context[:8000],  # ограничиваем длину контекста
        "answer": answer,
        "references": references
    })

df_v4 = pd.DataFrame(results_v4)
df_v4.to_csv("submission_dense_qe_fusion_v4.csv", index=False)
print("✅ submission_dense_qe_fusion_v4.csv saved successfully!")


100%|██████████| 50/50 [05:19<00:00,  6.39s/it]

✅ submission_dense_qe_fusion_v4.csv saved successfully!
